# Experiment: Darcy FNO Correction Visualization

读取 Darcy correction run 的 `config.json`、`predictions.mat`、`eval_metrics.json` 和 `correction_diagnostics.mat`，可视化：
- baseline backbone 与 corrected output 的对比
- `target_fine` 参考解
- `pred_corrected - target_fine` 与 `pred_backbone - target_fine`
- correction field `b_h`
- `|tau_x| + |tau_y|`
- `residual_corrected` 与 `flux_error_corrected`

该分支固定沿用 Stage 1 baseline Darcy-FNO，只新增 Stage 2 / Stage 3；训练时只使用训练集前 `100` 个样本，如果训练集更小则使用全部。


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


def find_repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().resolve().parents]:
        if (candidate / 'run_artifacts.py').exists() and (candidate / 'utilities3.py').exists():
            return candidate
    raise RuntimeError('Could not locate Bias_Aware_FNO repo root from the current working directory.')


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from utilities3 import MatReader

plt.rcParams['figure.figsize'] = (6, 5)
plt.rcParams['image.cmap'] = 'viridis'
plt.rcParams['axes.titlesize'] = 12


## Locate A Correction Run

In [ ]:
def latest_correction_run() -> Path | None:
    runs_root = REPO_ROOT / 'runs'
    matches = sorted(
        [p for p in runs_root.iterdir() if p.is_dir() and '_correction' in p.name and (p / 'config.json').exists()],
        key=lambda p: p.stat().st_mtime,
    )
    return matches[-1] if matches else None


RUN_DIR = latest_correction_run()
print('REPO_ROOT =', REPO_ROOT)
print('RUN_DIR   =', RUN_DIR)
assert RUN_DIR is not None, 'No correction run found under runs/'


## Load Artifacts

In [ ]:
CONFIG = json.loads((RUN_DIR / 'config.json').read_text(encoding='utf-8'))
EVAL_METRICS = json.loads((RUN_DIR / 'eval_metrics.json').read_text(encoding='utf-8'))

PRED = MatReader(str(RUN_DIR / 'predictions.mat'), to_torch=False)
DIAG = MatReader(str(RUN_DIR / 'correction_diagnostics.mat'), to_torch=False)
TEST = MatReader(str(CONFIG['test_path']), to_torch=False)

coeff = PRED.read_field('coeff')
pred_backbone = PRED.read_field('pred_backbone')
pred_corrected = PRED.read_field('pred_corrected')
target_coarse = PRED.read_field('target_coarse')
target_fine = PRED.read_field('target_fine')
residual_corrected = PRED.read_field('residual_corrected')
flux_error_corrected = PRED.read_field('flux_error_corrected')

b_h = DIAG.read_field('b_h')
tau_x = DIAG.read_field('tau_x')
tau_y = DIAG.read_field('tau_y')

mesh_coarse_points = TEST.read_field('mesh_coarse_points')
mesh_coarse_cells = TEST.read_field('mesh_coarse_cells')
mesh_fine_points = TEST.read_field('mesh_fine_points')
mesh_fine_cells = TEST.read_field('mesh_fine_cells')

print('coeff.shape            =', coeff.shape)
print('pred_backbone.shape    =', pred_backbone.shape)
print('pred_corrected.shape   =', pred_corrected.shape)
print('target_fine.shape      =', target_fine.shape)
print('b_h.shape              =', b_h.shape)
print('tau_x.shape / tau_y.shape =', tau_x.shape, tau_y.shape)


## Run Summary

In [ ]:
print('experiment_name           =', CONFIG['experiment_name'])
print('baseline_run_dir          =', CONFIG['extra']['backbone_run_dir'])
print('coeff_resolution          =', CONFIG['extra']['coeff_resolution'])
print('coarse_n / fine_n         =', CONFIG['extra']['coarse_n'], '/', CONFIG['extra']['fine_n'])
print('backbone_modes / width    =', CONFIG['extra']['backbone_modes'], '/', CONFIG['extra']['backbone_width'])
print('correction_modes / width  =', CONFIG['args']['correction_modes'], '/', CONFIG['args']['correction_width'])
print('correction_train_samples  =', CONFIG['extra']['correction_train_samples'])
print('stage2_epochs / stage3_epochs =', CONFIG['args']['stage2_epochs'], '/', CONFIG['args']['stage3_epochs'])
print('lambda_backbone / state / pde / flux / reg / mask =')
print(
    CONFIG['args']['lambda_backbone'],
    CONFIG['args']['lambda_state'],
    CONFIG['args']['lambda_pde'],
    CONFIG['args']['lambda_flux'],
    CONFIG['args']['lambda_reg'],
    CONFIG['args']['lambda_mask'],
)
print('--- eval metrics ---')
for key, value in EVAL_METRICS.items():
    print(f'{key:28s}: {value}')


## Sample-Level Error Statistics

In [ ]:
sample_fine_l2_backbone = np.linalg.norm((pred_backbone - target_fine).reshape(pred_backbone.shape[0], -1), axis=1) / np.clip(
    np.linalg.norm(target_fine.reshape(target_fine.shape[0], -1), axis=1), 1e-12, None
)
sample_fine_l2_corrected = np.linalg.norm((pred_corrected - target_fine).reshape(pred_corrected.shape[0], -1), axis=1) / np.clip(
    np.linalg.norm(target_fine.reshape(target_fine.shape[0], -1), axis=1), 1e-12, None
)
sample_flux_l2 = np.linalg.norm(flux_error_corrected.reshape(flux_error_corrected.shape[0], -1), axis=1)
improvement = (sample_fine_l2_backbone - sample_fine_l2_corrected) / np.clip(sample_fine_l2_backbone, 1e-12, None)

print('backbone fine L2 mean  =', float(sample_fine_l2_backbone.mean()))
print('corrected fine L2 mean =', float(sample_fine_l2_corrected.mean()))
print('improvement mean       =', float(improvement.mean()))
print('best improvement idx   =', int(np.argmax(improvement)))
print('worst improvement idx  =', int(np.argmin(improvement)))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].hist(sample_fine_l2_backbone, bins=20, alpha=0.75, label='backbone')
axes[0].hist(sample_fine_l2_corrected, bins=20, alpha=0.75, label='corrected')
axes[0].set_title('Fine Relative L2 Distribution')
axes[0].legend()

axes[1].plot(np.sort(sample_fine_l2_backbone), label='backbone')
axes[1].plot(np.sort(sample_fine_l2_corrected), label='corrected')
axes[1].set_title('Sorted Fine Relative L2')
axes[1].legend()

axes[2].hist(improvement, bins=20)
axes[2].set_title('Improvement Ratio Distribution')
axes[2].axvline(0.0, color='black', linestyle='--', linewidth=1)

fig.tight_layout()
plt.show()


## Single Sample Visualization

In [ ]:
sample_idx = int(np.argmin(sample_fine_l2_corrected))
print('sample_idx =', sample_idx)
print('backbone fine L2  =', float(sample_fine_l2_backbone[sample_idx]))
print('corrected fine L2 =', float(sample_fine_l2_corrected[sample_idx]))
print('improvement       =', float(improvement[sample_idx]))

abs_tau = np.zeros_like(coeff[sample_idx])
abs_tau[1:, :] += np.abs(tau_x[sample_idx])
abs_tau[:-1, :] += np.abs(tau_x[sample_idx])
abs_tau[:, 1:] += np.abs(tau_y[sample_idx])
abs_tau[:, :-1] += np.abs(tau_y[sample_idx])
abs_tau *= 0.5

fig, axes = plt.subplots(2, 5, figsize=(24, 9))
panels = [
    (coeff[sample_idx], 'Coefficient', 'viridis'),
    (pred_backbone[sample_idx], 'Backbone Prediction', 'RdBu_r'),
    (pred_corrected[sample_idx], 'Corrected Prediction', 'RdBu_r'),
    (target_fine[sample_idx], 'Fine Target', 'RdBu_r'),
    (target_coarse[sample_idx], 'Coarse Target', 'RdBu_r'),
    (pred_backbone[sample_idx] - target_fine[sample_idx], 'Backbone - Fine', 'RdBu_r'),
    (pred_corrected[sample_idx] - target_fine[sample_idx], 'Corrected - Fine', 'RdBu_r'),
    (b_h[sample_idx], 'Correction b_h', 'RdBu_r'),
    (residual_corrected[sample_idx], 'Residual Corrected', 'magma'),
    (flux_error_corrected[sample_idx], 'Flux Error Corrected', 'magma'),
]

for ax, (field, title, cmap) in zip(axes.ravel(), panels):
    im = ax.imshow(field, cmap=cmap)
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
    plt.colorbar(im, ax=ax, shrink=0.8)

fig.tight_layout()
plt.show()

plt.figure(figsize=(6, 5))
plt.imshow(abs_tau, cmap='magma')
plt.title('|tau_x| + |tau_y|')
plt.xticks([])
plt.yticks([])
plt.colorbar(shrink=0.8)
plt.show()


## Coarse / Fine FEM Meshes

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, points, cells, title in [
    (axes[0], mesh_coarse_points, mesh_coarse_cells, 'Coarse FEM Mesh'),
    (axes[1], mesh_fine_points, mesh_fine_cells, 'Fine FEM Mesh'),
]:
    ax.triplot(points[:, 0], points[:, 1], cells[: min(len(cells), 4000)], linewidth=0.3)
    ax.set_title(title)
    ax.set_aspect('equal')
    ax.set_xticks([])
    ax.set_yticks([])
fig.tight_layout()
plt.show()


## Worst-Case Gallery

In [ ]:
worst_indices = np.argsort(sample_fine_l2_corrected)[-3:][::-1]
fig, axes = plt.subplots(len(worst_indices), 4, figsize=(16, 4 * len(worst_indices)))
if len(worst_indices) == 1:
    axes = axes[None, :]
for row, idx in enumerate(worst_indices):
    views = [
        (pred_backbone[idx], f'Backbone #{idx}', 'RdBu_r'),
        (pred_corrected[idx], f'Corrected #{idx}', 'RdBu_r'),
        (target_fine[idx], f'Fine Target #{idx}', 'RdBu_r'),
        (pred_corrected[idx] - target_fine[idx], f'Error #{idx}', 'RdBu_r'),
    ]
    for ax, (field, title, cmap) in zip(axes[row], views):
        im = ax.imshow(field, cmap=cmap)
        ax.set_title(title)
        ax.set_xticks([])
        ax.set_yticks([])
        plt.colorbar(im, ax=ax, shrink=0.8)
fig.tight_layout()
plt.show()
